# 🚀 Titanic V2 — Modelo Melhorado para Score > 0.80
**InsurMinds | Iteração 2 — Engenharia Avançada + XGBoost + Ensemble**

---

## Objetivo desta Iteração
Partindo do score **0.78468** obtido na V1 (Random Forest), buscamos ultrapassar **0.80** através de:
1. **Feature Engineering Avançado** — FarePerPerson, TicketPrefix, AgeTitle
2. **XGBoost + LightGBM** — algoritmos de gradient boosting mais poderosos
3. **Ensemble por Votação (Voting Classifier)** — combinação inteligente de modelos
4. **Redução de Overfitting** — regularização e parâmetros controlados

## 1. Importações e Configuração

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Gradient Boosting Avançado
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Estilo visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.size'] = 11

SEED = 42
print('Bibliotecas importadas com sucesso!')
import xgboost, lightgbm, sklearn
print(f'XGBoost: {xgboost.__version__} | LightGBM: {lightgbm.__version__} | Scikit-Learn: {sklearn.__version__}')

## 2. Carregamento dos Dados

In [ ]:
TRAIN_PATH = '../dados/train.csv'
TEST_PATH  = '../dados/test.csv'

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f'Treino: {train_df.shape} | Teste: {test_df.shape}')
display(train_df.head(3))

## 3. Feature Engineering Avançado

### Novas Features desta iteração:
| Feature | Justificativa |
|---|---|
| `FarePerPerson` | Tarifa dividida pelo tamanho do grupo — proxy real do poder aquisitivo |
| `TicketPrefix` | Prefixo do bilhete indica setor/classe do navio |
| `FamilyCategory` | Solo / Pequena / Média / Grande — mais granular que IsAlone |
| `IsMother / IsChild` | Grupos com prioridade histórica de resgate |
| `DeckNum` | Convés como valor numérico ordinal |

In [ ]:
def feature_engineering_v2(df, ticket_counts_ref=None):
    df = df.copy()

    # 1. Tamanho da Família
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

    # 2. Viajando Sozinho
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    # 3. Título Social
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    title_mapping = {
        'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
        'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
        'Mlle': 'Miss', 'Mme': 'Mrs', 'Don': 'Rare', 'Lady': 'Rare',
        'Countess': 'Rare', 'Jonkheer': 'Rare', 'Sir': 'Rare',
        'Capt': 'Rare', 'Ms': 'Miss'
    }
    df['Title'] = df['Title'].map(title_mapping).fillna('Rare')

    # 4. Cabine Conhecida
    df['CabinKnown'] = df['Cabin'].notnull().astype(int)

    # 5. Convés
    df['Deck'] = df['Cabin'].apply(lambda s: s[0] if pd.notnull(s) else 'M')
    deck_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7, 'M': 0, 'T': 8}
    df['DeckNum'] = df['Deck'].map(deck_map).fillna(0).astype(int)

    # 6. Tamanho do Grupo por Bilhete
    if ticket_counts_ref is None:
        ticket_counts_ref = df['Ticket'].value_counts()
    df['TicketGroupSize'] = df['Ticket'].map(ticket_counts_ref).fillna(1).astype(int)

    # 7. Faixas Etárias (mais granulares)
    df['AgeGroup'] = pd.cut(
        df['Age'],
        bins=[-1, 5, 12, 18, 35, 60, 120],
        labels=['Baby', 'Child', 'Teen', 'YoungAdult', 'Adult', 'Senior']
    )
    df['AgeGroup'] = df['AgeGroup'].astype(str).replace('nan', 'Unknown')

    # 8. Faixas de Tarifa
    df['FareGroup'] = pd.qcut(
        df['Fare'].fillna(df['Fare'].median()), q=4,
        labels=['Low', 'Medium', 'High', 'VeryHigh']
    )
    df['FareGroup'] = df['FareGroup'].astype(str)

    # 9. FarePerPerson [NOVO]
    df['FarePerPerson'] = df['Fare'].fillna(df['Fare'].median()) / df['TicketGroupSize']

    # 10. TicketPrefix [NOVO]
    def extract_ticket_prefix(ticket):
        parts = ticket.split()
        if len(parts) > 1:
            return parts[0].replace('.', '').replace('/', '').upper()[:4]
        return 'NUM'
    df['TicketPrefix'] = df['Ticket'].apply(extract_ticket_prefix)
    top_prefixes = df['TicketPrefix'].value_counts().nlargest(10).index
    df['TicketPrefix'] = df['TicketPrefix'].where(df['TicketPrefix'].isin(top_prefixes), 'OTHER')

    # 11. FamilyCategory [NOVO]
    def family_category(size):
        if size == 1:   return 'Solo'
        if size <= 3:   return 'Small'
        if size <= 6:   return 'Medium'
        return 'Large'
    df['FamilyCategory'] = df['FamilySize'].apply(family_category)

    # 12. IsMother / IsChild [NOVO]
    df['Age_filled'] = df['Age'].fillna(df['Age'].median())
    df['IsMother'] = ((df['Sex'] == 'female') & (df['Parch'] > 0) & (df['Age_filled'] > 18)).astype(int)
    df['IsChild']  = (df['Age_filled'] < 12).astype(int)
    df.drop(columns=['Age_filled'], inplace=True)

    # 13. TitleNum — encoding ordinal de título
    title_num = {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4}
    df['TitleNum'] = df['Title'].map(title_num).fillna(4)

    return df, ticket_counts_ref


train_fe, ticket_counts = feature_engineering_v2(train_df)
test_fe,  _             = feature_engineering_v2(test_df, ticket_counts_ref=ticket_counts)

orig_cols = set(train_df.columns)
new_cols  = [c for c in train_fe.columns if c not in orig_cols]
print(f'Treino: {train_fe.shape} | Teste: {test_fe.shape}')
print(f'Novas features ({len(new_cols)}):', new_cols)

## 4. Preparação das Features

In [ ]:
DROP_COLS = ['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin']

X_train  = train_fe.drop([c for c in DROP_COLS if c in train_fe.columns], axis=1)
y_train  = train_fe['Survived']
X_test   = test_fe.drop([c for c in DROP_COLS if c in test_fe.columns and c != 'Survived'], axis=1)
test_ids = test_df['PassengerId']

numeric_features = [
    'Age', 'Fare', 'SibSp', 'Parch',
    'FamilySize', 'TicketGroupSize',
    'FarePerPerson', 'DeckNum', 'TitleNum',
    'IsAlone', 'CabinKnown', 'IsMother', 'IsChild'
]
categorical_features = [
    'Pclass', 'Sex', 'Embarked',
    'Title', 'Deck', 'AgeGroup', 'FareGroup',
    'FamilyCategory', 'TicketPrefix'
]

numeric_features     = [f for f in numeric_features     if f in X_train.columns]
categorical_features = [f for f in categorical_features if f in X_train.columns]

print(f'Features numéricas  ({len(numeric_features)}):',  numeric_features)
print(f'Features categóricas({len(categorical_features)}):', categorical_features)

## 5. Construção do Pré-processador (Pipeline)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

print('Preprocessor construído com sucesso!')

## 6. Treinamento e Comparação dos Modelos (5-Fold CV)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

lr_model   = LogisticRegression(random_state=SEED, max_iter=1000, C=0.1)
rf_model   = RandomForestClassifier(
    random_state=SEED, n_estimators=200, max_depth=6,
    min_samples_split=10, min_samples_leaf=4, max_features='sqrt'
)
xgb_model  = XGBClassifier(
    random_state=SEED, n_estimators=300, max_depth=3,
    learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    use_label_encoder=False, eval_metric='logloss', verbosity=0
)
lgbm_model = LGBMClassifier(
    random_state=SEED, n_estimators=300, max_depth=4,
    learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, num_leaves=15, verbose=-1
)
voting_model = VotingClassifier(
    estimators=[('lr', lr_model), ('rf', rf_model), ('xgb', xgb_model), ('lgbm', lgbm_model)],
    voting='soft'
)

models_to_eval = {
    'Regressao Logistica': lr_model,
    'Random Forest V1':    rf_model,
    'XGBoost':             xgb_model,
    'LightGBM':            lgbm_model,
    'Voting Ensemble':     voting_model
}

results = []
print('Avaliando modelos com 5-Fold CV...\n')

for name, model in models_to_eval.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    results.append({
        'Modelo': name,
        'Acuracia Media': scores.mean(),
        'Desvio Padrao':  scores.std(),
    })
    print(f'{name:<30}: {scores.mean():.4f} +/- {scores.std():.4f}')

results_df = pd.DataFrame(results).sort_values('Acuracia Media', ascending=False)
print('\nAvaliacao concluida!')

## 7. Visualização Comparativa

In [ ]:
display(results_df.style
    .highlight_max(subset=['Acuracia Media'], color='#adffc4')
    .highlight_min(subset=['Desvio Padrao'], color='#adf4ff')
    .format({'Acuracia Media': '{:.4f}', 'Desvio Padrao': '{:.4f}'})
)

fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#e74c3c', '#3498db', '#f39c12', '#2ecc71', '#9b59b6']
ax.barh(results_df['Modelo'], results_df['Acuracia Media'], color=colors[:len(results_df)], edgecolor='white')
ax.errorbar(
    results_df['Acuracia Media'], results_df['Modelo'],
    xerr=results_df['Desvio Padrao'],
    fmt='none', color='black', capsize=5, linewidth=2
)
ax.set_xlim(0.70, 0.90)
ax.axvline(0.78468, color='red',   linestyle='--', linewidth=1.5, label='Score V1 Kaggle (0.78468)')
ax.axvline(0.80000, color='green', linestyle='--', linewidth=1.5, label='Meta: 0.80000')
ax.legend(loc='lower right')
ax.set_title('Comparacao de Modelos V2 — Acuracia CV 5-Fold', fontsize=13, fontweight='bold')
ax.set_xlabel('Acuracia Media')
plt.tight_layout()
plt.show()

## 8. Importância das Features (XGBoost)

In [ ]:
xgb_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   XGBClassifier(
        random_state=SEED, n_estimators=300, max_depth=3,
        learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='logloss', verbosity=0
    ))
])
xgb_pipe.fit(X_train, y_train)

cat_enc   = xgb_pipe.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
cat_names = cat_enc.get_feature_names_out(categorical_features)
all_names = numeric_features + list(cat_names)

importances = xgb_pipe.named_steps['classifier'].feature_importances_
feat_df = pd.DataFrame({'Feature': all_names, 'Importancia': importances})
feat_df = feat_df.sort_values('Importancia', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(x='Importancia', y='Feature', data=feat_df, palette='viridis', ax=ax)
ax.set_title('Top 20 Features — XGBoost Feature Importance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Modelo Final e Geração da Submissão

In [ ]:
print('Treinando Voting Ensemble no conjunto completo de treino...')

final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   VotingClassifier(
        estimators=[
            ('lr',   LogisticRegression(random_state=SEED, max_iter=1000, C=0.1)),
            ('rf',   RandomForestClassifier(
                random_state=SEED, n_estimators=200, max_depth=6,
                min_samples_split=10, min_samples_leaf=4, max_features='sqrt'
            )),
            ('xgb',  XGBClassifier(
                random_state=SEED, n_estimators=300, max_depth=3,
                learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                reg_alpha=0.1, reg_lambda=1.0,
                use_label_encoder=False, eval_metric='logloss', verbosity=0
            )),
            ('lgbm', LGBMClassifier(
                random_state=SEED, n_estimators=300, max_depth=4,
                learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                reg_alpha=0.1, reg_lambda=1.0, num_leaves=15, verbose=-1
            ))
        ],
        voting='soft'
    ))
])

final_pipeline.fit(X_train, y_train)
test_predictions = final_pipeline.predict(X_test)

sobreviventes = test_predictions.sum()
total = len(test_predictions)
print(f'Predicoes geradas: {total} passageiros')
print(f'  Nao sobreviveram (0): {total - sobreviventes} ({(total-sobreviventes)/total*100:.1f}%)')
print(f'  Sobreviveram    (1): {sobreviventes} ({sobreviventes/total*100:.1f}%)')

submission = pd.DataFrame({'PassengerId': test_ids, 'Survived': test_predictions})

OUTPUT_PATH = '../submissao_titanic_v2.csv'
submission.to_csv(OUTPUT_PATH, index=False)

print(f'\nArquivo salvo em: {OUTPUT_PATH}')
display(submission.head(10))

## 10. Resumo da Iteração V2

| Item | V1 (Baseline) | V2 (Esta Iteração) |
|---|---|---|
| **Score Kaggle** | 0.78468 | 🎯 A ser medido |
| **Melhor CV Local** | 0.8384 (RF) | Voting Ensemble |
| **Algoritmos** | LR, DT, RF | LR, RF, XGB, LGBM, Voting |
| **Nº de Features** | 14 | 22+ |
| **FarePerPerson** | Nao | Sim |
| **TicketPrefix** | Nao | Sim |
| **FamilyCategory** | Nao | Sim |
| **IsMother / IsChild** | Nao | Sim |
| **DeckNum (numerico)** | Nao | Sim |
| **Ensemble (Voting)** | Nao | Sim |

### Arquivo de submissao gerado: `submissao_titanic_v2.csv`
> Utilize este arquivo para a proxima submissao no Kaggle.